# UD2 Conjunto de Datos stranger_things

## **Enunciado.**
El proyecto consiste en que hagas tu propio grep y wordcount sobre los datos que elegiste en la tarea anterior. Abarca diversos temas como la gestión de ficheros con HDFS, comandos de Unix y uso de MapReduce.

Debes entregar un cuaderno de Colab con:  
  * Explicación del conjunto de datos elegido: cuál es su temática, su tamaño, con qué campos se relaciona y por qué puede resultar interesante.
   
  * Ahora explica qué información quieres extraer con grep.  
  Por ejemplo, si usamos un Datasheet de Apple TV+ que consta de dos CSV (titles.csv y credits.csv). En este caso interesaría analizar el archivo credits.csv y contar las veces que aparece la palabra ACTOR y las que aparece DIRECTOR.

  * Una vez quede claro el contexto:  
    1. Escribe y explica de manera detallada todas las operaciones que usas para copiar en el sistema de archivos de HDFS los archivos.  
    2. Escribe los comandos necesarios para realizar el conteo mediante grep y extrae (y explica) tus conclusiones sobre los resultados.  
    3. Escribe un programa MapReduce en Java que cuente cuántas veces aparece cada palabra en cada uno de los ficheros.  
    4. Escribe un programa MapReduce en Python que a través de Hadoop-Streaming que cuente cuántas veces aparece cada palabra en cada uno de los ficheros.


# Mi Conjunto de Datos: `Stranger Things`

Para este proyecto, he elegido tres conjuntos de datos relacionados con la serie 'Stranger Things': `stranger_things_all_dialogue.csv`, `Stranger_Things_Ratings.csv` y `episodes.csv`.


*   **Temática**: Mis datos giran en torno a la serie 'Stranger Things'.
*   **Tamaño**: El de diálogos tiene unas 32,519 entradas, el de valoraciones 34 y el de episodios también 34.
*   **Relación entre campos**: Los tres se conectan por la `season` (temporada) y el `episode` (episodio), lo que me permite unir la información.
*   **Interés**: Me parece interesante analizar los diálogos y las tramas para entender la importancia de los personajes y conceptos clave en la serie.

# ¿Qué información quiero extraer con grep?

Para el análisis con `grep`, me centraré en el dataset `stranger_things_all_dialogue.csv`, que contiene los diálogos de la serie.

Mi objetivo es buscar y contar las apariciones de palabras clave para entender la relevancia de ciertos personajes o conceptos dentro de la narrativa. Específicamente, me interesa:

1.  **Contar las apariciones de nombres de personajes principales**: Así tendré una idea de la prominencia de personajes como **'Will'**, **'Hopper'** y **'Dustin'** a lo largo de los diálogos.
2.  **Contar la frecuencia de palabras temáticas**: Palabras como **'monster'** (monstruo), **'upside down'** (mundo del revés) o **'gate'** (portal) son cruciales para la trama y su frecuencia me puede indicar la intensidad de ciertos elementos argumentales en diferentes temporadas o episodios.

Esta información me dará una perspectiva inicial sobre la importancia de estos elementos en el contexto de la serie.

# Carga de Datasets CSV y Verificación Inicial

**Voy a cargar** los datasets `stranger_things_all_dialogue.csv`, `Stranger_Things_Ratings.csv` y `episodes.csv` en DataFrames de pandas para iniciar el proceso ETL. Es fundamental verificar que la carga se haga bien para asegurar la integridad de los datos.

## Carga de Datos desde el Ordenador (Alternativa Interactiva)

Aunque **ya subí** los archivos directamente, existe una forma interactiva de cargar archivos desde mi ordenador durante la ejecución del cuaderno de Colab. Esto me es útil cuando no tengo una URL pública o una API, y necesito interactuar con el entorno para seleccionar y subir un archivo localmente.

In [25]:
from google.colab import files
import io

print("Por favor, selecciona el archivo 'dataset' desde tu ordenador:")
uploaded = files.upload()

# Solo voy a cargar un archivo a la vez para este ejemplo
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  df_dialogue_uploaded = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
  print("\nPrimeras 5 filas del DataFrame cargado desde el ordenador ('df_dialogue_uploaded'):")
  display(df_dialogue_uploaded.head())

Por favor, selecciona el archivo 'dataset' desde tu ordenador:


## Ejemplo Clonar repo ejemplo para cargar dataset

In [1]:
!git clone https://github.com/kachytronico/BDA_examen_26

Cloning into 'BDA_examen_26'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 63 (delta 16), reused 52 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 928.42 KiB | 10.55 MiB/s, done.
Resolving deltas: 100% (16/16), done.


## Cargar y comprobar sus primeras filas.

In [2]:
import pandas as pd

df_dialogue = pd.read_csv('/content/BDA_examen_26/Datasets_ejemplos_examen/Datasets strangerthings/stranger_things_all_dialogue.csv')
print("Primeras 5 filas del DataFrame 'df_dialogue':")
display(df_dialogue.head())
print("-" * 30)

df_ratings = pd.read_csv('/content/BDA_examen_26/Datasets_ejemplos_examen/Datasets strangerthings/Stranger_Things_Ratings.csv')
print("Primeras 5 filas del DataFrame 'df_ratings':")
display(df_ratings.head())
print("-" * 30)

df_episodes = pd.read_csv('/content/BDA_examen_26/Datasets_ejemplos_examen/Datasets strangerthings/episodes.csv')
print("Primeras 5 filas del DataFrame 'df_episodes':")
display(df_episodes.head())

Primeras 5 filas del DataFrame 'df_dialogue':


,season,episode,line,raw_text,stage_direction,dialogue,start_time,end_time
0,1,1,1,[crickets chirping],[crickets chirping],NaN,00:00:07,00:00:09
1,1,1,2,[alarm blaring],[alarm blaring],NaN,00:00:49,00:00:51
2,1,1,3,[panting],[panting],NaN,00:00:52,00:00:54
3,1,1,4,[elevator descending],[elevator descending],NaN,00:01:01,00:01:02
4,1,1,5,[elevator dings],[elevator dings],NaN,00:01:09,00:01:10


------------------------------
Primeras 5 filas del DataFrame 'df_ratings':


,index,Episode_Number,Title,Image_url,Year,Description,Genre,Runtime,Rating
0,0,1,Chapter One: The Vanishing of Will Byers,https://m.mediaamazon.com/images/M/MV5BMTUwNTE...,2016,At the U.S. Dept. of Energy an unexplained eve...,Drama | Fantasy | Horror,47 min,8.6
1,1,2,Chapter Two: The Weirdo on Maple Street,https://m.mediaamazon.com/images/M/MV5BMjA4NzA...,2016,Mike hides the mysterious girl in his house. J...,Drama | Fantasy | Horror,55 min,8.5
2,2,3,Chapter Three: Holly Jolly,https://m.mediaamazon.com/images/M/MV5BOTkyMDQ...,2016,An increasingly concerned Nancy looks for Barb...,Drama | Fantasy | Horror,51 min,8.9
3,3,4,Chapter Four: The Body,https://m.mediaamazon.com/images/M/MV5BMTkwMjU...,2016,Refusing to believe Will is dead Joyce tries ...,Drama | Fantasy | Horror,49 min,9.0
4,4,5,Chapter Five: The Flea and the Acrobat,https://m.mediaamazon.com/images/M/MV5BMjM0NjQ...,2016,Hopper breaks into the lab to find the truth a...,Drama | Fantasy | Horror,52 min,8.8


------------------------------
Primeras 5 filas del DataFrame 'df_episodes':


,season,episode,title,directed_by,written_by,original_release_date
0,1,1,Chapter One: The Vanishing of Will Byers,The Duffer Brothers,The Duffer Brothers,2016-07-15
1,1,2,Chapter Two: The Weirdo on Maple Street,The Duffer Brothers,The Duffer Brothers,2016-07-15
2,1,3,"Chapter Three: Holly, Jolly",Shawn Levy,Jessica Mecklenburg,2016-07-15
3,1,4,Chapter Four: The Body,Shawn Levy,Justin Doble,2016-07-15
4,1,5,Chapter Five: The Flea and the Acrobat,The Duffer Brothers,Alison Tatlock,2016-07-15


# 2. Perfilado de Datos del `df_dialogue`

Antes de buscar o analizar, necesito entender bien la estructura y el contenido de mi conjunto de datos principal: `stranger_things_all_dialogue.csv` (cargado como `df_dialogue`). Este perfilado me ayuda a identificar columnas importantes, tipos de datos y posibles problemas como valores nulos.

In [3]:
print('--- Primeras 5 filas del DataFrame df_dialogue ---')
display(df_dialogue.head())

--- Primeras 5 filas del DataFrame df_dialogue ---


,season,episode,line,raw_text,stage_direction,dialogue,start_time,end_time
0,1,1,1,[crickets chirping],[crickets chirping],NaN,00:00:07,00:00:09
1,1,1,2,[alarm blaring],[alarm blaring],NaN,00:00:49,00:00:51
2,1,1,3,[panting],[panting],NaN,00:00:52,00:00:54
3,1,1,4,[elevator descending],[elevator descending],NaN,00:01:01,00:01:02
4,1,1,5,[elevator dings],[elevator dings],NaN,00:01:09,00:01:10


In [4]:
print('\n--- Información general del DataFrame df_dialogue ---')
df_dialogue.info()


--- Información general del DataFrame df_dialogue ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32519 entries, 0 to 32518
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   season           32519 non-null  int64 
 1   episode          32519 non-null  int64 
 2   line             32519 non-null  int64 
 3   raw_text         32519 non-null  object
 4   stage_direction  10678 non-null  object
 5   dialogue         26435 non-null  object
 6   start_time       32519 non-null  object
 7   end_time         32519 non-null  object
dtypes: int64(3), object(5)
memory usage: 2.0+ MB


In [5]:
print('\n--- Estadísticas descriptivas del DataFrame df_dialogue ---')
display(df_dialogue.describe(include='all'))


--- Estadísticas descriptivas del DataFrame df_dialogue ---


,season,episode,line,raw_text,stage_direction,dialogue,start_time,end_time
count,32519.000000,32519.000000,32519.000000,32519,10678,26435,32519,32519
unique,NaN,NaN,NaN,26832,4106,22819,5631,5626
top,NaN,NaN,NaN,[sighs],[sighs],,00:11:54,00:26:36
freq,NaN,NaN,NaN,146,283,394,19,20
mean,2.856853,4.998032,545.291214,NaN,NaN,NaN,NaN,NaN
std,1.122935,2.530884,396.296415,NaN,NaN,NaN,NaN,NaN
min,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
25%,2.000000,3.000000,240.000000,NaN,NaN,NaN,NaN,NaN
50%,3.000000,5.000000,479.000000,NaN,NaN,NaN,NaN,NaN
75%,4.000000,7.000000,757.000000,NaN,NaN,NaN,NaN,NaN


In [6]:
print('\n--- Conteo de valores nulos por columna en df_dialogue ---')
display(df_dialogue.isnull().sum())


--- Conteo de valores nulos por columna en df_dialogue ---


,0
season,0
episode,0
line,0
raw_text,0
stage_direction,21841
dialogue,6084
start_time,0
end_time,0


El perfilado de datos de `df_dialogue` me muestra que la columna `raw_text` es la más relevante para mis búsquedas tipo `grep`. También observo valores nulos en algunas columnas como `speaker` y `stage_direction`, algo importante para futuras limpiezas o análisis.

# Hadoop

Lo primero que hago es activar el soporte de GPU.

**"Entorno de ejecución" > "Cambiar tipo de entorno de ejecución"**

## Verificaciones previas

Primero, verifico la versión de Java:

In [7]:
!ls -l /usr/lib/jvm/

total 4
lrwxrwxrwx 1 root root   21 Jan 29 03:35 java-1.17.0-openjdk-amd64 -> java-17-openjdk-amd64
drwxr-xr-x 9 root root 4096 Apr  2 13:13 java-17-openjdk-amd64


## Instalación de Hadoop

Entro a https://downloads.apache.org/hadoop/common y veo las versiones disponibles para descargar.

Una vez comprobado qué versión de Java necesito para la versión de Hadoop que usaré, descargo el tar.gz correspondiente.

Para saberlo, consulto el archivo Changelog y busco "Java". Ahí veo frases como:

"[JDK17] Add JPMS options required by Java 17+"

In [8]:
!wget https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz

--2026-04-22 11:50:17--  https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz
Resolving downloads.apache.org (downloads.apache.org)... 88.99.208.237, 135.181.214.104, 2a01:4f9:3a:2c57::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|88.99.208.237|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1065831750 (1016M) [application/x-gzip]
Saving to: ‘hadoop-3.4.2.tar.gz’

hadoop-3.4.2.tar.gz 100%[===================>]   1016M  31.0MB/s    in 34s     

2026-04-22 11:50:52 (29.7 MB/s) - ‘hadoop-3.4.2.tar.gz’ saved [1065831750/1065831750]



Una vez descargado, lo extraigo en el sistema de archivos de Colab.

In [9]:
!tar -xzf hadoop-3.4.2.tar.gz

Luego, muevo la carpeta extraída a /usr/local

In [10]:
!mv  hadoop-3.4.2/ /usr/local/hadoop

## Configuración

Actualizo las variables de entorno (JAVA_HOME, PATH).

In [11]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/usr/local/hadoop"
os.environ["PATH"] += f":{os.environ['HADOOP_HOME']}/bin:{os.environ['HADOOP_HOME']}/sbin"

Compruebo la instalación.

In [12]:
!hadoop version

Hadoop 3.4.2
Source code repository https://github.com/apache/hadoop.git -r 84e8b89ee2ebe6923691205b9e171badde7a495c
Compiled by ahmarsu on 2025-08-20T10:30Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum fa94c67d4b4be021b9e9515c9b0f7b6
This command was run using /usr/local/hadoop/share/hadoop/common/hadoop-common-3.4.2.jar


# 1.- HDFS

Creo un directorio "STUD2" (el nombre viene de Stranger Things Unidad 2)

In [13]:
!hdfs dfs -mkdir STUD2

Compruebo que se ha creado bien con el comando "ls"

In [14]:
!hdfs dfs -ls

Found 5 items
drwxr-xr-x   - root root       4096 2026-04-16 13:28 .config
drwxr-xr-x   - root root       4096 2026-04-22 11:50 BDA_examen_26
drwxr-xr-x   - root root       4096 2026-04-22 11:51 STUD2
-rw-r--r--   1 root root 1065831750 2025-10-04 09:30 hadoop-3.4.2.tar.gz
drwxr-xr-x   - root root       4096 2026-04-16 13:28 sample_data


Aquí, el sistema de archivos HDFS es el mismo que el local, así que no necesito el comando `put` para gestionarlo (en Google Colab no existe). De todas formas, mostraré los comandos que usaría en un entorno donde sí fuera necesario.

## Obtener los .csv

In [15]:
import os

output_dir = '/content/drive/MyDrive/Colab_Notebooks/BDA/UD2'
os.makedirs(output_dir, exist_ok=True)

# Guardo el DataFrame de diálogos
output_file_dialogue = os.path.join(output_dir, 'stranger_things_dialogue.csv')
df_dialogue.to_csv(output_file_dialogue, index=False)
print(f"DataFrame de diálogos guardado en: {output_file_dialogue}")

DataFrame de diálogos guardado en: /content/drive/MyDrive/Colab_Notebooks/BDA/UD2/stranger_things_dialogue.csv


En un caso real, usaría el comando `PUT` de Hadoop para transferir los archivos entre nodos, copiando desde mi sistema local al destino. Como mencioné antes, en Google Colab no es necesario.

In [16]:
!hdfs dfs -put /content/drive/MyDrive/Colab_Notebooks/BDA/UD2/*.csv STUD2

Ahora, verifico el contenido de la carpeta `STUD2` en HDFS:

In [17]:
!hdfs dfs -ls STUD2

Found 1 items
-rw-r--r--   1 root root    2822680 2026-04-22 11:51 STUD2/stranger_things_dialogue.csv


# 2.- Realizando MapReduce Grep para 'Will' en el dataset de diálogos

Aunque los archivos están en `/STUD2` en HDFS, los copio a mi disco local (`/tmp/hadoop_input`) para que el programa `grep` de ejemplo los procese bien.

En un clúster Hadoop de producción, no necesitaría este paso extra de `hdfs dfs -get`, y apuntaría `hadoop jar` directamente a las rutas de HDFS.

In [18]:
LOCAL_BASE_DIR = '/content/STUD2'
LOCAL_INPUT_DIR = f'{LOCAL_BASE_DIR}/local_hadoop_input'
LOCAL_OUTPUT_DIR = f'{LOCAL_BASE_DIR}/local_hadoop_output'

# Crear directorios de entrada y salida locales si no existen
!mkdir -p {LOCAL_INPUT_DIR}
!mkdir -p {LOCAL_OUTPUT_DIR}

# Eliminar cualquier copia local anterior del CSV de diálogos en la nueva carpeta
!rm -f {LOCAL_INPUT_DIR}/stranger_things_dialogue.csv

# Copiar el CSV de diálogos de HDFS /STUD2 al sistema de archivos local
!hdfs dfs -get STUD2/stranger_things_dialogue.csv {LOCAL_INPUT_DIR}/stranger_things_dialogue.csv

# Eliminar la salida anterior de grep local si existe para evitar errores
!rm -rf {LOCAL_OUTPUT_DIR}/grep_will_results_dialogues

# Ejecutar el trabajo MapReduce Grep en el archivo local
!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep \
    {LOCAL_INPUT_DIR}/stranger_things_dialogue.csv \
    {LOCAL_OUTPUT_DIR}/grep_will_results_dialogues \
    'Will'

2026-04-22 11:51:36,038 INFO input.FileInputFormat: Total input files to process : 1
2026-04-22 11:51:36,089 INFO mapreduce.JobSubmitter: number of splits:1
2026-04-22 11:51:36,567 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local396966913_0001
2026-04-22 11:51:36,569 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-22 11:51:36,873 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-04-22 11:51:36,875 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-04-22 11:51:36,875 INFO mapreduce.Job: Running job: job_local396966913_0001
2026-04-22 11:51:36,889 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFactory
2026-04-22 11:51:36,891 INFO output.FileOutputCommitter: File Output Committer Algorithm version is 2
2026-04-22 11:51:36,891 INFO output.FileOutputCommitter: FileOutputCommitter skip cleanup _temporary folders under output directory:false, ignore cleanup f

### Listo el contenido del directorio de resultados

In [19]:
LOCAL_BASE_DIR = '/content/STUD2'
LOCAL_OUTPUT_DIR = f'{LOCAL_BASE_DIR}/local_hadoop_output'

# Listar el contenido del directorio de resultados local
!ls -l {LOCAL_OUTPUT_DIR}/grep_will_results_dialogues

total 4
-rw-r--r-- 1 root root 9 Apr 22 11:51 part-r-00000
-rw-r--r-- 1 root root 0 Apr 22 11:51 _SUCCESS


### Muestro los resultados del MapReduce Grep

In [20]:
LOCAL_BASE_DIR = '/content/STUD2'
LOCAL_OUTPUT_DIR = f'{LOCAL_BASE_DIR}/local_hadoop_output'

# Mostrar los resultados del MapReduce Grep desde el archivo de salida local
!cat {LOCAL_OUTPUT_DIR}/grep_will_results_dialogues/part-r-00000

984	Will


Paso los archivos a HDFS dentro de la carpeta STUD2:

In [21]:
!hdfs dfs -put /content/STUD2/local_hadoop_output/grep_will_results_dialogues STUD2/

Visualizo de forma recursiva el contenido de la carpeta en HDFS.

In [22]:
!hdfs dfs -ls -R STUD2

drwxr-xr-x   - root root       4096 2026-04-22 11:51 STUD2/local_hadoop_input
-rw-r--r--   1 root root    2822680 2026-04-22 11:51 STUD2/local_hadoop_input/stranger_things_dialogue.csv
drwxr-xr-x   - root root       4096 2026-04-22 11:51 STUD2/grep_will_results_dialogues
-rw-r--r--   1 root root          9 2026-04-22 11:51 STUD2/grep_will_results_dialogues/part-r-00000
-rw-r--r--   1 root root          0 2026-04-22 11:51 STUD2/grep_will_results_dialogues/_SUCCESS
-rw-r--r--   1 root root    2822680 2026-04-22 11:51 STUD2/stranger_things_dialogue.csv
drwxr-xr-x   - root root       4096 2026-04-22 11:51 STUD2/local_hadoop_output
drwxr-xr-x   - root root       4096 2026-04-22 11:51 STUD2/local_hadoop_output/grep_will_results_dialogues
-rw-r--r--   1 root root          9 2026-04-22 11:51 STUD2/local_hadoop_output/grep_will_results_dialogues/part-r-00000
-rw-r--r--   1 root root          0 2026-04-22 11:51 STUD2/local_hadoop_output/grep_will_results_dialogues/_SUCCESS


### Conclusión detallada del Punto 1: Realizando el Grep de MapReduce para 'Will'

Aquí, mi objetivo fue usar el `grep` de MapReduce de Hadoop para encontrar y contar las veces que aparece la palabra 'Will' en el dataset `stranger_things_dialogue.csv`.

Para ello, seguí estos pasos:

1.  **Preparación para la Ejecución del Grep**: Aunque mi archivo `stranger_things_dialogue.csv` ya estaba en HDFS dentro de `STUD2`, necesité copiarlo primero a una ruta local para que el `grep` de Hadoop funcionara bien. Para eso, creé un directorio local (`/content/STUD2/local_hadoop_input`) y usé el comando `!hdfs dfs -get STUD2/stranger_things_dialogue.csv {LOCAL_INPUT_DIR}/stranger_things_dialogue.csv`.

2.  **Ejecución del Conteo mediante Grep con MapReduce**: Una vez que el archivo estuvo disponible localmente, ejecuté el trabajo `grep` de MapReduce. Usé el comando `!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep`, indicando como entrada el archivo local (`{LOCAL_INPUT_DIR}/stranger_things_dialogue.csv`) y como salida un nuevo directorio también en el sistema de archivos local (`{LOCAL_OUTPUT_DIR}/grep_will_results_dialogues`). Este comando me permitió buscar 'Will' y contar sus apariciones.

3.  **Inspección y Consolidación de Resultados**: Para ver los resultados, listé el contenido del directorio de salida local (`!ls -l {LOCAL_OUTPUT_DIR}/grep_will_results_dialogues`) y luego usé `!cat` para ver el archivo `part-r-00000`. Finalmente, para que los resultados estuvieran en HDFS, copié el directorio de resultados local de vuelta a `STUD2` en HDFS con `!hdfs dfs -put /content/STUD2/local_hadoop_output/grep_will_results_dialogues STUD2/`.

**Conclusión sobre los resultados**: Al ver el archivo `part-r-00000`, encontré la línea `984 Will`. Esto me dice que el nombre 'Will' aparece 984 veces en los diálogos de `stranger_things_all_dialogue.csv`. Este conteo me da una idea clara de lo importante que es este personaje en la serie, lo cual es muy útil para mi análisis de datos y cumple mi objetivo de sacar información clave del tema.

## 2.2.- Realizando MapReduce Grep para 'Hopper' en el dataset de diálogos

In [23]:
import os

LOCAL_BASE_DIR = '/content/STUD2'
LOCAL_INPUT_DIR = f'{LOCAL_BASE_DIR}/local_hadoop_input'
LOCAL_OUTPUT_DIR = f'{LOCAL_BASE_DIR}/local_hadoop_output'

# Crear directorio de salida local si no existe
!mkdir -p {LOCAL_OUTPUT_DIR}

# Eliminar la salida anterior de grep local para 'Hopper' si existe para evitar errores
!rm -rf {LOCAL_OUTPUT_DIR}/grep_hopper_results_dialogues

# Ejecutar el trabajo MapReduce Grep en el archivo local para 'Hopper'
!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep \
    {LOCAL_INPUT_DIR}/stranger_things_dialogue.csv \
    {LOCAL_OUTPUT_DIR}/grep_hopper_results_dialogues \
    'Hopper'

2026-04-22 11:51:48,047 INFO input.FileInputFormat: Total input files to process : 1
2026-04-22 11:51:48,094 INFO mapreduce.JobSubmitter: number of splits:1
2026-04-22 11:51:48,443 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local9725417_0001
2026-04-22 11:51:48,445 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-22 11:51:48,735 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-04-22 11:51:48,735 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-04-22 11:51:48,736 INFO mapreduce.Job: Running job: job_local9725417_0001
2026-04-22 11:51:48,748 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFactory
2026-04-22 11:51:48,750 INFO output.FileOutputCommitter: File Output Committer Algorithm version is 2
2026-04-22 11:51:48,750 INFO output.FileOutputCommitter: FileOutputCommitter skip cleanup _temporary folders under output directory:false, ignore cleanup failu

### Listo el contenido del directorio de resultados para 'Hopper'

In [24]:
# Listar el contenido del directorio de resultados
!ls -l {LOCAL_OUTPUT_DIR}/grep_hopper_results_dialogues

total 4
-rw-r--r-- 1 root root 11 Apr 22 11:51 part-r-00000
-rw-r--r-- 1 root root  0 Apr 22 11:51 _SUCCESS


### Muestro los resultados del MapReduce Grep para 'Hopper'

In [25]:
# Mostrar los resultados del MapReduce Grep
!cat {LOCAL_OUTPUT_DIR}/grep_hopper_results_dialogues/part-r-00000

626	Hopper


### Conclusión sobre el Grep de 'Hopper'

El resultado del `grep` para 'Hopper' es `626 Hopper`. Esto me dice que el nombre 'Hopper' aparece 626 veces en los diálogos. Este conteo, aunque menor de lo que esperaba, es importante y destaca la relevancia de Hopper en la historia de 'Stranger Things'.

## 2.3.- Realizando MapReduce Grep para 'Dustin' en el dataset de diálogos

In [26]:
# Eliminar la salida anterior de grep para 'Dustin'
!rm -rf {LOCAL_OUTPUT_DIR}/grep_dustin_results_dialogues

# Ejecutar el trabajo MapReduce Grep para 'Dustin'
!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep \
    {LOCAL_INPUT_DIR}/stranger_things_dialogue.csv \
    {LOCAL_OUTPUT_DIR}/grep_dustin_results_dialogues \
    'Dustin'

2026-04-22 11:51:53,689 INFO input.FileInputFormat: Total input files to process : 1
2026-04-22 11:51:53,729 INFO mapreduce.JobSubmitter: number of splits:1
2026-04-22 11:51:54,062 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1604143722_0001
2026-04-22 11:51:54,064 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-22 11:51:54,314 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-04-22 11:51:54,315 INFO mapreduce.Job: Running job: job_local1604143722_0001
2026-04-22 11:51:54,318 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-04-22 11:51:54,327 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFactory
2026-04-22 11:51:54,328 INFO output.FileOutputCommitter: File Output Committer Algorithm version is 2
2026-04-22 11:51:54,328 INFO output.FileOutputCommitter: FileOutputCommitter skip cleanup _temporary folders under output directory:false, ignore cleanup

### Listo el contenido del directorio de resultados para 'Dustin'

In [27]:
# Listar el contenido del directorio de resultados
!ls -l {LOCAL_OUTPUT_DIR}/grep_dustin_results_dialogues

total 4
-rw-r--r-- 1 root root 11 Apr 22 11:51 part-r-00000
-rw-r--r-- 1 root root  0 Apr 22 11:51 _SUCCESS


### Muestro los resultados del MapReduce Grep para 'Dustin'

In [28]:
# Mostrar los resultados del MapReduce Grep
!cat {LOCAL_OUTPUT_DIR}/grep_dustin_results_dialogues/part-r-00000

852	Dustin


### Conclusión sobre el Grep de 'Dustin'

El resultado del `grep` para 'Dustin' es `852 Dustin`. Esto significa que el nombre 'Dustin' aparece 852 veces en los diálogos. Comparado con 'Will' (984) y 'Hopper' (626), Dustin tiene una presencia considerable en los diálogos, lo que encaja con su papel activo en la trama.

## 2.4.- Realizando MapReduce Grep para 'monster' en el dataset de diálogos

In [29]:
# Eliminar la salida anterior de grep para 'monster'
!rm -rf {LOCAL_OUTPUT_DIR}/grep_monster_results_dialogues

# Ejecutar el trabajo MapReduce Grep para 'monster'
!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep \
    {LOCAL_INPUT_DIR}/stranger_things_dialogue.csv \
    {LOCAL_OUTPUT_DIR}/grep_monster_results_dialogues \
    'monster'

2026-04-22 11:52:00,237 INFO input.FileInputFormat: Total input files to process : 1
2026-04-22 11:52:00,284 INFO mapreduce.JobSubmitter: number of splits:1
2026-04-22 11:52:00,629 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local339882476_0001
2026-04-22 11:52:00,631 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-22 11:52:00,883 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-04-22 11:52:00,883 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-04-22 11:52:00,884 INFO mapreduce.Job: Running job: job_local339882476_0001
2026-04-22 11:52:00,897 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFactory
2026-04-22 11:52:00,898 INFO output.FileOutputCommitter: File Output Committer Algorithm version is 2
2026-04-22 11:52:00,899 INFO output.FileOutputCommitter: FileOutputCommitter skip cleanup _temporary folders under output directory:false, ignore cleanup f

### Listo el contenido del directorio de resultados para 'monster'

In [30]:
# Listar el contenido del directorio de resultados
!ls -l {LOCAL_OUTPUT_DIR}/grep_monster_results_dialogues

total 4
-rw-r--r-- 1 root root 12 Apr 22 11:52 part-r-00000
-rw-r--r-- 1 root root  0 Apr 22 11:52 _SUCCESS


### Muestro los resultados del MapReduce Grep para 'monster'

In [31]:
# Mostrar los resultados del MapReduce Grep
!cat {LOCAL_OUTPUT_DIR}/grep_monster_results_dialogues/part-r-00000

234	monster


### Conclusión sobre el Grep de 'monster'

El resultado del `grep` para 'monster' es `234 monster`. Esto me indica que la palabra 'monster' aparece 234 veces en los diálogos. Aunque no es tan frecuente como los nombres de los personajes principales, esta cifra muestra la amenaza constante y el punto clave de la trama que son las criaturas del "Upside Down".

## 2.5.- Realizando MapReduce Grep para 'upside down' en el dataset de diálogos

In [32]:
# Eliminar la salida anterior de grep para 'upside down'
!rm -rf {LOCAL_OUTPUT_DIR}/grep_upside_down_results_dialogues

# Ejecutar el trabajo MapReduce Grep para 'upside down'
!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep \
    {LOCAL_INPUT_DIR}/stranger_things_dialogue.csv \
    {LOCAL_OUTPUT_DIR}/grep_upside_down_results_dialogues \
    'upside down'

2026-04-22 11:52:05,809 INFO input.FileInputFormat: Total input files to process : 1
2026-04-22 11:52:05,842 INFO mapreduce.JobSubmitter: number of splits:1
2026-04-22 11:52:06,185 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local377977146_0001
2026-04-22 11:52:06,187 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-22 11:52:06,458 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-04-22 11:52:06,458 INFO mapreduce.Job: Running job: job_local377977146_0001
2026-04-22 11:52:06,459 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-04-22 11:52:06,471 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFactory
2026-04-22 11:52:06,472 INFO output.FileOutputCommitter: File Output Committer Algorithm version is 2
2026-04-22 11:52:06,472 INFO output.FileOutputCommitter: FileOutputCommitter skip cleanup _temporary folders under output directory:false, ignore cleanup f

### Listo el contenido del directorio de resultados para 'upside down'

In [33]:
# Listar el contenido del directorio de resultados
!ls -l {LOCAL_OUTPUT_DIR}/grep_upside_down_results_dialogues

total 4
-rw-r--r-- 1 root root 14 Apr 22 11:52 part-r-00000
-rw-r--r-- 1 root root  0 Apr 22 11:52 _SUCCESS


### Muestro los resultados del MapReduce Grep para 'upside down'

In [34]:
# Mostrar los resultados del MapReduce Grep
!cat {LOCAL_OUTPUT_DIR}/grep_upside_down_results_dialogues/part-r-00000

6	upside down


### Conclusión sobre el Grep de 'upside down'

El resultado del `grep` para la frase 'upside down' es `6 upside down`. Esto significa que la frase clave 'upside down' aparece 6 veces en los diálogos. Esta frecuencia, aunque baja, subraya la importancia de esta dimensión paralela como algo central y misterioso en la trama. Su poca aparición podría indicar momentos clave donde se habla de su apertura o cierre, haciendo que cada mención sea importante.

## 2.6.- Realizando MapReduce Grep para 'gate' en el dataset de diálogos

In [35]:
# Eliminar la salida anterior de grep para 'gate'
!rm -rf {LOCAL_OUTPUT_DIR}/grep_gate_results_dialogues

# Ejecutar el trabajo MapReduce Grep para 'gate'
!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep \
    {LOCAL_INPUT_DIR}/stranger_things_dialogue.csv \
    {LOCAL_OUTPUT_DIR}/grep_gate_results_dialogues \
    'gate'

2026-04-22 11:52:12,894 INFO input.FileInputFormat: Total input files to process : 1
2026-04-22 11:52:12,928 INFO mapreduce.JobSubmitter: number of splits:1
2026-04-22 11:52:13,271 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1135896034_0001
2026-04-22 11:52:13,273 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-22 11:52:13,583 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-04-22 11:52:13,584 INFO mapreduce.Job: Running job: job_local1135896034_0001
2026-04-22 11:52:13,585 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-04-22 11:52:13,597 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFactory
2026-04-22 11:52:13,598 INFO output.FileOutputCommitter: File Output Committer Algorithm version is 2
2026-04-22 11:52:13,598 INFO output.FileOutputCommitter: FileOutputCommitter skip cleanup _temporary folders under output directory:false, ignore cleanup

### Listo el contenido del directorio de resultados para 'gate'

In [36]:
# Listar el contenido del directorio de resultados
!ls -l {LOCAL_OUTPUT_DIR}/grep_gate_results_dialogues

total 4
-rw-r--r-- 1 root root 9 Apr 22 11:52 part-r-00000
-rw-r--r-- 1 root root 0 Apr 22 11:52 _SUCCESS


### Muestro los resultados del MapReduce Grep para 'gate'

In [37]:
# Mostrar los resultados del MapReduce Grep
!cat {LOCAL_OUTPUT_DIR}/grep_gate_results_dialogues/part-r-00000

218	gate


### Conclusión sobre el Grep de 'gate'

El resultado del `grep` para 'gate' es `218 gate`. Esto me indica que la palabra 'gate' aparece 218 veces en los diálogos. Esta frecuencia es clave, porque representa un concepto fundamental en la mitología de la serie: el punto de conexión entre las dimensiones.

# 3.- Programa MapReduce en Java para WordCount

Para este punto, necesito implementar un programa MapReduce en Java que cuente la frecuencia de cada palabra en mi conjunto de datos. Este proceso tiene varios pasos: preparo el entorno, escribo el código Java para el `Mapper`, `Reducer` y la clase principal `Driver`, compilo el código, lo empaqueto en un archivo JAR y finalmente lo ejecuto con Hadoop.

## 3.1. Preparación del Entorno y Estructura de Directorios

Primero, creo una estructura de directorios para mi proyecto Java. Esto me ayuda a organizar los archivos fuente y a compilar más fácilmente.

In [38]:
import os

JAVA_PROJECT_DIR = '/content/wordcount_java'
JAVA_SRC_DIR = f'{JAVA_PROJECT_DIR}/src'

# Crear los directorios
!mkdir -p {JAVA_SRC_DIR}
print(f"Directorio de proyecto Java creado en: {JAVA_PROJECT_DIR}")
print(f"Directorio de código fuente Java creado en: {JAVA_SRC_DIR}")

Directorio de proyecto Java creado en: /content/wordcount_java
Directorio de código fuente Java creado en: /content/wordcount_java/src


## 3.2. Código Fuente Java para WordCount

Ahora, escribo las clases Java para mi programa WordCount. Estas son:

*   **`WordCountMapper.java`**: Extiende `Mapper` y procesa el texto de entrada, emitiendo pares (palabra, 1).
*   **`WordCountReducer.java`**: Extiende `Reducer` y agrega los conteos de las palabras, emitiendo pares (palabra, conteo total).
*   **`WordCount.java`**: La clase principal (Driver) que configura y ejecuta el trabajo MapReduce.

### 3.2.1. `WordCountMapper.java`

In [39]:
%%writefile {JAVA_SRC_DIR}/WordCountMapper.java
import java.io.IOException;
import java.util.StringTokenizer;

import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Mapper;

public class WordCountMapper extends Mapper<Object, Text, Text, IntWritable> {

    // NORMALMENTE NO CAMBIAR: constante para emitir 1 por cada ocurrencia
    private final static IntWritable one = new IntWritable(1);

    // NORMALMENTE NO CAMBIAR: objeto reutilizable para la clave de salida
    private Text word = new Text();

    // MAPREDUCE llama a este método una vez por cada línea de entrada
    @Override
    public void map(Object key, Text value, Context context) throws IOException, InterruptedException {

        // CAMBIAR CASI SIEMPRE:
        // aquí decides cómo separar o interpretar cada línea del fichero
        // Para WordCount clásico usamos StringTokenizer sobre espacios
        StringTokenizer itr = new StringTokenizer(value.toString());

        while (itr.hasMoreTokens()) {

            // CAMBIAR CASI SIEMPRE:
            // si el ejercicio pide limpiar símbolos, pasar a minúsculas,
            // quitar comas, ignorar números o saltar cabecera,
            // esta es una de las zonas donde suele hacerse
            String token = itr.nextToken().trim();

            // REVISAR:
            // si quieres ignorar tokens vacíos o basura, filtra aquí
            if (!token.isEmpty()) {
                word.set(token);

                // CAMBIAR CASI SIEMPRE:
                // esta línea define qué pareja (clave, valor) emite el mapper
                // En WordCount emitimos (palabra, 1)
                context.write(word, one);
            }
        }
    }
}


Writing /content/wordcount_java/src/WordCountMapper.java


### 3.2.2. `WordCountReducer.java`

In [40]:
%%writefile {JAVA_SRC_DIR}/WordCountReducer.java
import java.io.IOException;

import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Reducer;

public class WordCountReducer extends Reducer<Text, IntWritable, Text, IntWritable> {

    // NORMALMENTE NO CAMBIAR: objeto reutilizable para la salida
    private IntWritable result = new IntWritable();

    // MAPREDUCE llama a reduce una vez por cada clave única
    @Override
    public void reduce(Text key, Iterable<IntWritable> values, Context context)
            throws IOException, InterruptedException {

        int sum = 0;

        // NORMALMENTE NO CAMBIAR en WordCount:
        // sumar todos los valores asociados a la misma palabra
        for (IntWritable val : values) {
            sum += val.get();
        }

        // CAMBIAR SOLO SI el ejercicio pide otra agregación:
        // por ejemplo máximo, media, mínimo, concatenación, etc.
        result.set(sum);

        // NORMALMENTE NO CAMBIAR:
        // emitimos (palabra, total)
        context.write(key, result);
    }
}

Writing /content/wordcount_java/src/WordCountReducer.java


### 3.2.3. `WordCount.java` (Driver)

In [41]:
%%writefile {JAVA_SRC_DIR}/WordCount.java
import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class WordCount {

    public static void main(String[] args) throws Exception {

        // REVISAR:
        // Hadoop espera normalmente dos argumentos:
        // args[0] = ruta de entrada
        // args[1] = ruta de salida
        if (args.length != 2) {
            System.err.println("Uso: WordCount <input path> <output path>");
            System.exit(-1);
        }

        Configuration conf = new Configuration();

        // CAMBIAR:
        // nombre del trabajo que se verá en Hadoop
        Job job = Job.getInstance(conf, "word count");

        // CAMBIAR:
        // debe coincidir con el nombre de esta clase principal
        job.setJarByClass(WordCount.class);

        // CAMBIAR:
        // si cambias el nombre del mapper o reducer, cámbialo aquí también
        job.setMapperClass(WordCountMapper.class);

        // REVISAR:
        // el combiner suele poder ser el mismo reducer en WordCount
        // si el ejercicio usa otra lógica, comprueba si sigue siendo válido
        job.setCombinerClass(WordCountReducer.class);
        job.setReducerClass(WordCountReducer.class);

        // REVISAR:
        // estos tipos deben coincidir con la salida final del reducer
        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(IntWritable.class);

        // CAMBIAR SIEMPRE EN EJECUCIÓN:
        // ruta de entrada y ruta de salida en HDFS
        FileInputFormat.addInputPath(job, new Path(args[0]));
        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}


Writing /content/wordcount_java/src/WordCount.java


## 3.3. Compilación del Código Java

Ahora, compilo el código Java usando `javac`. Necesito incluir las librerías de Hadoop en el classpath para que la compilación funcione bien.

In [42]:
JAVA_CLASSES_DIR = f'{JAVA_PROJECT_DIR}/classes'
!mkdir -p {JAVA_CLASSES_DIR}

# Obtener las rutas de los JARs de Hadoop necesarios para el classpath
HADOOP_CLASSPATH = !find /usr/local/hadoop/share/hadoop -name "*.jar" | tr '\n' ':'
HADOOP_CLASSPATH = HADOOP_CLASSPATH[0].strip(':')

!javac -classpath {HADOOP_CLASSPATH} -d {JAVA_CLASSES_DIR} {JAVA_SRC_DIR}/*.java

print(f"Clases compiladas en: {JAVA_CLASSES_DIR}")

Clases compiladas en: /content/wordcount_java/classes


## 3.4. Creación del Archivo JAR

Una vez compiladas las clases, las empaqueto en un archivo JAR, que es el formato que Hadoop espera para ejecutar los trabajos MapReduce.

In [43]:
JAVA_JAR_FILE = f'{JAVA_PROJECT_DIR}/wordcount.jar'

# Crear el archivo JAR a partir de las clases compiladas
!jar -cvf {JAVA_JAR_FILE} -C {JAVA_CLASSES_DIR} .

print(f"Archivo JAR creado en: {JAVA_JAR_FILE}")

added manifest
adding: WordCount.class(in = 1558) (out= 876)(deflated 43%)
adding: WordCountMapper.class(in = 1775) (out= 793)(deflated 55%)
adding: WordCountReducer.class(in = 1697) (out= 720)(deflated 57%)
Archivo JAR creado en: /content/wordcount_java/wordcount.jar


## 3.5. Ejecución del Programa MapReduce con Hadoop

Ahora, ejecuto mi trabajo WordCount usando el comando `hadoop jar`. La entrada será mi archivo de diálogos (que ya está en HDFS en `STUD2/stranger_things_dialogue.csv`) y especifico un directorio de salida en HDFS.

In [44]:
HDFS_INPUT_PATH_WC = 'STUD2/stranger_things_dialogue.csv'
HDFS_OUTPUT_PATH_WC = 'STUD2/wordcount_output_java'

# Eliminar el directorio de salida de Hadoop si ya existe para evitar errores
!hdfs dfs -rm -r {HDFS_OUTPUT_PATH_WC}

# Ejecutar el trabajo WordCount MapReduce
!hadoop jar {JAVA_JAR_FILE} WordCount {HDFS_INPUT_PATH_WC} {HDFS_OUTPUT_PATH_WC}

print(f"Trabajo WordCount ejecutado. Resultados en HDFS: {HDFS_OUTPUT_PATH_WC}")

rm: `STUD2/wordcount_output_java': No such file or directory
2026-04-22 11:52:28,727 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-04-22 11:52:28,857 INFO input.FileInputFormat: Total input files to process : 1
2026-04-22 11:52:28,889 INFO mapreduce.JobSubmitter: number of splits:1
2026-04-22 11:52:29,217 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local276468544_0001
2026-04-22 11:52:29,219 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-22 11:52:29,522 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-04-22 11:52:29,522 INFO mapreduce.Job: Running job: job_local276468544_0001
2026-04-22 11:52:29,525 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-04-22 11:52:29,535 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFact

## 3.6. Visualización de los Resultados

Finalmente, recupero y muestro los resultados del trabajo WordCount desde HDFS.

In [45]:
# Listar el contenido del directorio de salida en HDFS
!hdfs dfs -ls {HDFS_OUTPUT_PATH_WC}

print("\nPrimeras 20 líneas del resultado del WordCount en Java:")
!hdfs dfs -cat {HDFS_OUTPUT_PATH_WC}/part-r-00000 | head -n 20

Found 2 items
-rw-r--r--   1 root root          0 2026-04-22 11:52 STUD2/wordcount_output_java/_SUCCESS
-rw-r--r--   1 root root    1963697 2026-04-22 11:52 STUD2/wordcount_output_java/part-r-00000

Primeras 20 líneas del resultado del WordCount en Java:
""18	2
""99	2
""A	2
""America""	2
""Bad	1
""Be	2
""Big	2
""Blip.""",00:06:33,00:06:34	1
""But...""",,"Cocky	1
""But...""",00:28:04,00:28:06	1
""Byers.""",,"The	1
""Byers.""",00:55:35,00:55:37	1
""Cheeanti.""",,"I	1
""Cheeanti.""",00:05:12,00:05:14	1
""Close	1
""Compromise""?",[stutters],"""Compromise""?",00:08:29,00:08:32	1
""Department	2
""Don't	2
""Do…	2
""Dream	4
cat: Unable to write to output stream.


## Conclusión del Programa MapReduce en Java

El programa MapReduce en Java se ejecutó bien, dándome un conteo detallado de la frecuencia de cada palabra en el dataset de diálogos. Este proceso, aunque más complejo de implementar que el `grep` de Hadoop, me muestra el poder de MapReduce para procesar grandes volúmenes de datos de forma distribuida. La salida me muestra pares de (palabra, frecuencia), lo que me permite analizar la distribución de palabras y entender mejor el contenido del dataset.

# 4.- Programa MapReduce en Python con Hadoop Streaming

Para este punto, implemento un programa WordCount usando Python y Hadoop Streaming. Esto significa crear dos scripts de Python: uno para el `Mapper` y otro para el `Reducer`, que Hadoop ejecutará para procesar el `stranger_things_dialogue.csv`.

## 4.1. Preparación del Entorno y Scripts de Python

Primero, creo un directorio para mis scripts de Python y luego los defino.

In [46]:
import os

PYTHON_PROJECT_DIR = '/content/wordcount_python'

# Crear los directorios
!mkdir -p {PYTHON_PROJECT_DIR}
print(f"Directorio de proyecto Python creado en: {PYTHON_PROJECT_DIR}")

Directorio de proyecto Python creado en: /content/wordcount_python


## 4.2. `mapper.py`

El script del `mapper` lee cada línea del archivo de entrada, la divide en palabras y emite cada palabra seguida de un '1' (que representa una ocurrencia).

In [47]:
%%writefile {PYTHON_PROJECT_DIR}/mapper.py
#!/usr/bin/env python3
import sys

# CAMBIAR CASI SIEMPRE:
# aquí decides cómo leer y separar cada línea de entrada
# En WordCount clásico, dividimos por espacios
for line in sys.stdin:
    # REVISAR:
    # si el fichero tiene cabecera y quieres saltarla,
    # esta es una zona típica para hacerlo
    line = line.strip()

    # CAMBIAR CASI SIEMPRE:
    # si el ejercicio pide limpiar comas, puntos, símbolos,
    # pasar a minúsculas o filtrar palabras, suele hacerse aquí
    words = line.split()

    for word in words:
        word = word.strip()

        # REVISAR:
        # si quieres ignorar tokens vacíos o basura, filtra aquí
        if word:
            # CAMBIAR CASI SIEMPRE:
            # esta salida define la pareja (clave, valor)
            # En WordCount emitimos: palabra\t1
            print(f"{word}\t1")


Writing /content/wordcount_python/mapper.py


## 4.3. `reducer.py`

El script del `reducer` lee las salidas ordenadas del `mapper` (pares `palabra\t1`), agrupa por palabra y suma los conteos para cada palabra única.

In [48]:
%%writefile {PYTHON_PROJECT_DIR}/reducer.py
#!/usr/bin/env python3
import sys

# NORMALMENTE NO CAMBIAR en WordCount:
# reducer acumulador clásico para sumar ocurrencias por clave
current_word = None
current_count = 0

for line in sys.stdin:
    line = line.strip()

    # REVISAR:
    # si alguna línea no viene con el formato esperado clave\tvalor,
    # aquí puedes decidir ignorarla
    try:
        word, count = line.split("\t", 1)
        count = int(count)
    except ValueError:
        continue

    if current_word == word:
        current_count += count
    else:
        if current_word is not None:
            # NORMALMENTE NO CAMBIAR:
            # emitimos la palabra anterior con su suma total
            print(f"{current_word}\t{current_count}")

        current_word = word
        current_count = count

# NO OLVIDAR:
# al salir del bucle hay que imprimir la última clave acumulada
if current_word is not None:
    print(f"{current_word}\t{current_count}")

Writing /content/wordcount_python/reducer.py


## 4.4. Ejecución del Programa MapReduce en Python con Hadoop Streaming

Ahora, ejecuto mi trabajo WordCount de Python usando `hadoop streaming`. Uso el archivo de diálogos (`stranger_things_dialogue.csv`) como entrada y especifico un directorio de salida en HDFS.

In [49]:
HDFS_INPUT_PATH_WC_PY = 'STUD2/stranger_things_dialogue.csv'
HDFS_OUTPUT_PATH_WC_PY = 'STUD2/wordcount_output_python'

# Eliminar el directorio de salida de Hadoop si ya existe para evitar errores
!hdfs dfs -rm -r {HDFS_OUTPUT_PATH_WC_PY}

# Ejecutar el trabajo WordCount MapReduce con Python Streaming
!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.2.jar \
    -input {HDFS_INPUT_PATH_WC_PY} \
    -output {HDFS_OUTPUT_PATH_WC_PY} \
    -mapper "python3 {PYTHON_PROJECT_DIR}/mapper.py" \
    -reducer "python3 {PYTHON_PROJECT_DIR}/reducer.py" \
    -file {PYTHON_PROJECT_DIR}/mapper.py \
    -file {PYTHON_PROJECT_DIR}/reducer.py

print(f"Trabajo WordCount en Python ejecutado. Resultados en HDFS: {HDFS_OUTPUT_PATH_WC_PY}")

rm: `STUD2/wordcount_output_python': No such file or directory
2026-04-22 11:52:41,280 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [/content/wordcount_python/mapper.py, /content/wordcount_python/reducer.py] [] /tmp/streamjob10459875168936552906.jar tmpDir=null
2026-04-22 11:52:42,215 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-04-22 11:52:42,490 INFO mapred.FileInputFormat: Total input files to process : 1
2026-04-22 11:52:42,519 INFO mapreduce.JobSubmitter: number of splits:1
2026-04-22 11:52:42,873 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local205674472_0001
2026-04-22 11:52:42,875 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-22 11:52:43,283 INFO mapred.LocalDistributedCacheManager: Localized file:/content/wordcount_python/mapper.py as file:/tmp/hadoop-root/mapred/local/job_local205674472_0001_213c4937-ade0-4a7a-b13b-08179bb1c1ef/mapper.py
2026-

## 4.5. Visualización de los Resultados del WordCount en Python

Finalmente, recupero y muestro los resultados del trabajo WordCount desde HDFS.

In [50]:
# Listar el contenido del directorio de salida en HDFS
!hdfs dfs -ls {HDFS_OUTPUT_PATH_WC_PY}

print("\nPrimeras 20 líneas del resultado del WordCount en Python:")
!hdfs dfs -cat {HDFS_OUTPUT_PATH_WC_PY}/part-00000 | head -n 20

Found 2 items
-rw-r--r--   1 root root          0 2026-04-22 11:52 STUD2/wordcount_output_python/_SUCCESS
-rw-r--r--   1 root root    1963697 2026-04-22 11:52 STUD2/wordcount_output_python/part-00000

Primeras 20 líneas del resultado del WordCount en Python:
""18	2
""99	2
""A	2
""America""	2
""Bad	1
""Be	2
""Big	2
""Blip.""",00:06:33,00:06:34	1
""But...""",,"Cocky	1
""But...""",00:28:04,00:28:06	1
""Byers.""",,"The	1
""Byers.""",00:55:35,00:55:37	1
""Cheeanti.""",,"I	1
""Cheeanti.""",00:05:12,00:05:14	1
""Close	1
""Compromise""?",[stutters],"""Compromise""?",00:08:29,00:08:32	1
""Department	2
""Don't	2
""Do…	2
""Dream	4
cat: Unable to write to output stream.


## Conclusión del Programa MapReduce en Python con Hadoop Streaming

He realizado con éxito el programa WordCount en Python usando Hadoop Streaming. Esto me ha permitido obtener un conteo de la frecuencia de cada palabra en los diálogos de Stranger Things, confirmando la utilidad de Hadoop Streaming para el procesamiento de texto.

# Conclusión Final

He completado con éxito todas las fases del proyecto. He gestionado ficheros en HDFS, realizado búsquedas con `grep` para extraer información clave de los diálogos de Stranger Things, y he implementado y ejecutado programas MapReduce de WordCount tanto en Java como en Python con Hadoop Streaming. Estoy satisfecho con los resultados obtenidos en cada etapa.